## FAISS

- Meat에서 개발한 벡터 유사도 검색 라이브러리
- cpu, gpu

In [1]:
# !pip install langchain langchain-core langchain-community faiss-cpu langchain-groq langchain-ollama

In [2]:
from langchain_community.document_loaders import TextLoader  # 텍스트 로더 모듈을 가져옵니다.
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document  # 문서 클래스를 가져옵니다.
import json  # json 모듈을 가져옵니다.

# 텍스트 로더를 생성합니다.
text_loader = TextLoader(
    file_path='data/qna_data.json',  # 데이터가 저장된 json파일의 경로입니다.
    encoding='utf-8'  # 파일이 utf-8 형식으로 읽어올려면 설정해야 합니다.
)

# 텍스트 로더를 호출하여 문서를 불러옵니다.
text_docs = text_loader.load()

# 첫 번째 문서의 페이지 내용을 json 문자열로 변환합니다.
json_doc_str = json.loads(text_docs[0].page_content)
print(json_doc_str)  # json 데이터가 출력됩니다.

# json 데이터에서 문서 목록을 생성합니다.
docs = [
    Document(  # Document 클래스를 사용하여 문서를 생성합니다.
        page_content=f"질문: {doc['question']}\n\n답변: {doc['answer']}",  # 페이지 내용은 질문과 답변으로 구성됩니다.
        metadata={  # 문서의 메타데이터를 지정합니다.
            'id': doc['id'],  # id는 json 데이터에서 가져옵니다.
            'category': doc['category'],  # category도 json 데이터에서 가져옵니다.
            'keywords': doc['keywords']  # keywords도 json 데이터에서 가져옵니다.
        }
    )
    for doc in json_doc_str  # json 데이터의 각 문서에 대해 반복합니다.
]

print(docs)  # 생성된 문서 목록이 출력됩니다.

C:\Users\Admin\AppData\Local\Temp\ipykernel_24544\3808885268.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader  # 텍스트 로더 모듈을 가져옵니다.


[{'id': 'qna_001', 'category': '배송', 'question': '배송은 얼마나 걸리나요?', 'answer': '일반 배송의 경우 주문 후 2-3일 내 배송됩니다. 제주도 및 도서산간 지역은 추가 1-2일이 소요될 수 있습니다. 새벽 배송 서비스를 이용하시면 당일 오전 7시 이전에 받아보실 수 있습니다. 배송 상태는 마이페이지의 주문조회에서 실시간으로 확인 가능합니다.', 'keywords': ['배송기간', '배송일', '소요시간', '새벽배송', '도서산간']}, {'id': 'qna_002', 'category': '반품', 'question': '상품 반품은 어떻게 하나요?', 'answer': '상품 수령 후 7일 이내에 반품 신청이 가능합니다. 마이페이지 > 주문내역에서 반품 신청을 하시면 됩니다. 단, 상품 택과 라벨이 훼손되지 않은 미사용 제품에 한하며, 식품이나 개봉한 화장품 등 일부 상품은 반품이 제한될 수 있습니다. 반품 배송비는 고객 변심의 경우 왕복 배송비 6,000원이 차감됩니다.', 'keywords': ['반품', '반품신청', '반품기간', '반품조건', '배송비']}, {'id': 'qna_003', 'category': '결제', 'question': '어떤 결제 방법을 사용할 수 있나요?', 'answer': '신용카드, 체크카드, 계좌이체, 무통장입금, 네이버페이, 카카오페이, 토스페이, 페이코 등 다양한 결제 수단을 제공합니다. 50만원 이상 구매 시 무이자 할부 혜택도 있으며, 법인카드로도 결제 가능합니다. 결제 완료 후 영수증은 마이페이지에서 출력하실 수 있습니다.', 'keywords': ['결제', '결제수단', '카드', '페이', '할부', '무이자']}, {'id': 'qna_004', 'category': '쿠폰', 'question': '쿠폰은 어떻게 사용하나요?', 'answer': '보유하신 쿠폰은 마이페이지 > 쿠폰함에서 확인하실 수 있습니다. 결제 페이지에서 쿠폰 적용하기 

In [18]:
from langchain_ollama.embeddings import OllamaEmbeddings

embedder = OllamaEmbeddings(model="nomic-embed-text")

vector_store = FAISS.from_documents(
    documents = docs,
    embedding=embedder
)

print(f'입력 문서 수: {len(docs)}')
print(f'저장된 벡터 수: {vector_store.index.ntotal}')
print(f'벡터 차원: {vector_store.index.d}')

입력 문서 수: 10
저장된 벡터 수: 10
벡터 차원: 768


In [6]:
vector_store.save_local('faiss_index') # 임베딩 결과

In [9]:
load_vectorstore=vector_store.load_local(
    'faiss_index',
    embeddings=embedder,
    allow_dangerous_deserialization=True
)

In [14]:
user_query = '제품이 제가 원하는게 아니에요. 취소나 환불 해주시는 거죠?'

# 유사도검색
results = load_vectorstore.similarity_search(
    query=user_query,
    k=3
)

for result in results:
    print(result.page_content)

질문: 주문을 취소하고 싶은데 어떻게 하나요?

답변: 상품 발송 전까지는 마이페이지 > 주문내역에서 직접 취소하실 수 있습니다. 이미 배송이 시작된 경우에는 상품 수령 후 반품 절차를 진행하셔야 합니다. 무통장 입금으로 결제하신 경우 입금 전이라면 자동 취소되며, 입금 후에는 2-3일 내 환불 처리됩니다. 카드 결제는 취소 승인 후 카드사에 따라 3-7일 소요됩니다.
질문: 쿠폰은 어떻게 사용하나요?

답변: 보유하신 쿠폰은 마이페이지 > 쿠폰함에서 확인하실 수 있습니다. 결제 페이지에서 쿠폰 적용하기 버튼을 클릭하여 사용 가능한 쿠폰을 선택하시면 자동으로 할인이 적용됩니다. 쿠폰은 최소 구매 금액 및 사용 조건이 있으며, 상품 쿠폰과 장바구니 쿠폰은 중복 사용이 가능합니다.
질문: 어떤 결제 방법을 사용할 수 있나요?

답변: 신용카드, 체크카드, 계좌이체, 무통장입금, 네이버페이, 카카오페이, 토스페이, 페이코 등 다양한 결제 수단을 제공합니다. 50만원 이상 구매 시 무이자 할부 혜택도 있으며, 법인카드로도 결제 가능합니다. 결제 완료 후 영수증은 마이페이지에서 출력하실 수 있습니다.
